In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../Raw/6-dimensions-for-website-2015-08-16.csv", delimiter=";")

df["pdi"] = pd.to_numeric(df["pdi"], errors="coerce")
df["idv"] = pd.to_numeric(df["idv"], errors="coerce")
df["mas"] = pd.to_numeric(df["mas"], errors="coerce")
df["uai"] = pd.to_numeric(df["uai"], errors="coerce")
df["ltowvs"] = pd.to_numeric(df["ltowvs"], errors="coerce")
df["ivr"] = pd.to_numeric(df["ivr"], errors="coerce")

df["iso3"] = df["ctr"]

df

,ctr,country,pdi,idv,mas,uai,ltowvs,ivr,iso3
0,AFE,Africa East,64.0,27.0,41.0,52.0,32.0,40.0,AFE
1,AFW,Africa West,77.0,20.0,46.0,54.0,9.0,78.0,AFW
2,ALB,Albania,NaN,NaN,NaN,NaN,61.0,15.0,ALB
3,ALG,Algeria,NaN,NaN,NaN,NaN,26.0,32.0,ALG
4,AND,Andorra,NaN,NaN,NaN,NaN,NaN,65.0,AND
...,...,...,...,...,...,...,...,...,...
106,URU,Uruguay,61.0,36.0,38.0,100.0,26.0,53.0,URU
107,VEN,Venezuela,81.0,12.0,73.0,76.0,16.0,100.0,VEN
108,VIE,Vietnam,70.0,20.0,40.0,30.0,57.0,35.0,VIE
109,ZAM,Zambia,NaN,NaN,NaN,NaN,30.0,42.0,ZAM


In [3]:
df.isna().mean().sort_values(ascending=False)

pdi        0.297297
mas        0.297297
idv        0.297297
uai        0.297297
ltowvs     0.135135
ivr        0.126126
ctr        0.000000
country    0.000000
iso3       0.000000
dtype: float64

In [4]:
def hofstede_distance(iso3_i, iso3_j, data):
    row1 = data[data["iso3"] == iso3_i].iloc[0]
    row2 = data[data["iso3"] == iso3_j].iloc[0]
    
    distance = np.sqrt(
        (row1["pdi"] - row2["pdi"])**2 +
        (row1["idv"] - row2["idv"])**2 +
        (row1["mas"] - row2["mas"])**2 +
        (row1["uai"] - row2["uai"])**2 +
        (row1["ltowvs"] - row2["ltowvs"])**2 +
        (row1["ivr"] - row2["ivr"])**2
    )
    
    return distance

In [5]:
from itertools import product
from tqdm import tqdm

iso3_codes = df["iso3"].unique()
pairs = list(product(iso3_codes, repeat=2))

distances = []
for iso3_i, iso3_j in tqdm(pairs):
    dist = hofstede_distance(iso3_i, iso3_j, df)
    distances.append((iso3_i, iso3_j, dist))

dist_df = pd.DataFrame(distances, columns=["iso3_i", "iso3_j", "d_cul"])
dist_df

100%|██████████| 12321/12321 [00:18<00:00, 654.80it/s]


,iso3_i,iso3_j,d_cul
0,AFE,AFE,0.000000
1,AFE,AFW,47.116876
2,AFE,ALB,NaN
3,AFE,ALG,NaN
4,AFE,AND,NaN
...,...,...,...
12316,ZIM,URU,NaN
12317,ZIM,VEN,NaN
12318,ZIM,VIE,NaN
12319,ZIM,ZAM,NaN


In [17]:
dist_df.isna().mean()

iso3_i    0.00000
iso3_j    0.00000
d_cul     0.65709
dtype: float64

In [16]:
pairs = [
    ["USA", "SWE"],
    ["FRA", "SWE"],
    ["POL", "SWE"],
    ]

for iso3_i, iso3_j in pairs:
    try:
        dist = hofstede_distance(iso3_i, iso3_j, df)
        print(f"Distance between {iso3_i} and {iso3_j}: {dist}")
    except:
        pass

Distance between USA and SWE: 69.62758074211685
Distance between FRA and SWE: 84.03570669661795
Distance between POL and SWE: 108.13417591122614


In [ ]:
# Save to clean folder
dist_df.to_csv("../Clean/hofstede_distances.csv", index=False)